In [ ]:
!curl -L -o ./creditcardfraud.zip\
  https://www.kaggle.com/api/v1/datasets/download/mlg-ulb/creditcardfraud

!unzip -o ./creditcardfraud.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 65.9M  100 65.9M    0     0  18.7M      0  0:00:03  0:00:03 --:--:-- 24.2M
Archive:  ./creditcardfraud.zip
  inflating: creditcard.csv          


In [ ]:
!pip install dask[dataframe] distributed
!pip install featurewiz==0.6.0
!pip install Pillow==9.0.0
!pip install xlrd —ignore-installed — no-deps
!pip install executing>0.10.0

ERROR: Invalid requirement: '—': Expected package name at the start of dependency specifier
    —
    ^


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 9.1 MB/s eta 0:00:00


In [ ]:
!pip install imbalanced-learn

In [ ]:
import os
num_cores = os.cpu_count()
print(f"Número de núcleos de CPU disponíveis: {num_cores}")

Número de núcleos de CPU disponíveis: 8


In [ ]:
import numpy as np
import pandas as pd
import featurewiz as fw
from featurewiz import FeatureWiz
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import (SMOTE, ADASYN, BorderlineSMOTE, KMeansSMOTE, SVMSMOTE)
from imblearn.under_sampling import (EditedNearestNeighbours, TomekLinks, CondensedNearestNeighbour,
                                    ClusterCentroids, RepeatedEditedNearestNeighbours, AllKNN,
                                    InstanceHardnessThreshold, NearMiss, NeighbourhoodCleaningRule,
                                    OneSidedSelection, RandomUnderSampler)
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import (HistGradientBoostingClassifier,VotingClassifier,StackingClassifier,
                              BaggingClassifier,AdaBoostClassifier,GradientBoostingClassifier,
                              IsolationForest,RandomForestClassifier)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import f1_score, classification_report
import joblib
import warnings
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.utils import resample
from sklearn.utils import shuffle
import psutil
import time


# Configurar para ignorar warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)


# 1. Função para remoção de outliers
def remove_outliers(X, y, iqr_multiplier=3.0, strategy='cap', quantil=0.25):
  """Remove ou ajusta outliers em X e ajusta y se necessário."""
  X = X.copy()
  if isinstance(X, pd.DataFrame):
      numeric_cols = X.select_dtypes(include=np.number).columns
  else:
      numeric_cols = range(X.shape[1])
      X = pd.DataFrame(X)

  lower_bounds = {}
  upper_bounds = {}

  # Calcular limites
  for col in numeric_cols:
      q1 = X[col].quantile(quantil)
      q3 = X[col].quantile(1.0-quantil)
      iqr = q3 - q1
      lower_bounds[col] = q1 - iqr_multiplier * iqr
      upper_bounds[col] = q3 + iqr_multiplier * iqr

  # Aplicar estratégia
  if strategy == 'remove':
      mask = pd.Series(True, index=X.index)
      for col in numeric_cols:
          col_mask = (X[col] >= lower_bounds[col]) & (X[col] <= upper_bounds[col])
          mask = mask & col_mask
      X = X[mask]
      y = y[mask]

  elif strategy == 'cap':
      for col in numeric_cols:
          X[col] = np.clip(X[col], lower_bounds[col], upper_bounds[col])

  return X, y



# 2. Função para executar Featurewiz
def apply_featurewiz(X_train, y_train, X_test, corr_limit=0.7, feature_engg='', category_encoders='', auto_encoders='', imbalanced=False):
    """Executa Featurewiz nos dados de treino e aplica a mesma transformação nos dados de teste"""
    try:
        # Criar dataframe temporário com target
        #data = pd.DataFrame(X_train).reset_index(drop=True)
        #data = pd.concat([data, y_train.reset_index(drop=True)], axis=1)

        y_train = pd.Series(y_train)

        # Executar Featurewiz + correção do uso do FeatureWiz
        fwiz = FeatureWiz(corr_limit=corr_limit,
            feature_engg=feature_engg,
            category_encoders=category_encoders,
            auto_encoders=auto_encoders,
            imbalanced=imbalanced,
            verbose=0)

        X_train_selected, y_treino = fwiz.fit_transform(X_train, y_train)
        X_test_selected = fwiz.transform(X_test)
        ### get list of selected features ###
        print(f"Features: {fwiz.features}")

        X_train_trans = X_train_selected
        X_test_trans = X_test_selected

        '''# Executar Featurewiz
        X_train_trans, X_test_trans = fw.featurewiz(
            data,
            target='Class',
            test_data = X_test, #send the test data to get the test transformed dataset
            corr_limit=corr_limit,
            feature_engg=feature_engg,
            category_encoders=category_encoders,
            auto_encoders=auto_encoders,
            imbalanced=imbalanced,
            verbose=0
        )'''


        # Remover coluna target dos dados transformados
        X_train_trans = X_train_trans.drop('Class', axis=1, errors='ignore')
        X_test_trans = X_test_trans.drop('Class', axis=1, errors='ignore')


        # Garantir que temos as mesmas colunas em ambos os conjuntos
        common_cols = [col for col in X_train_trans.columns if col in X_test_trans.columns]
        X_train_trans = X_train_trans[common_cols]
        X_test_trans = X_test_trans[common_cols]

        return X_train_trans, X_test_trans, y_treino, common_cols  # Retorna features comuns + correção para retornar o y modificado


    except Exception as e:
        print(f"Erro no Featurewiz: {e}")
        # Fallback: retornar dados originais
        return X_train, X_test, y_train, X_train.columns.tolist()



def balance_dataset(X, y, k_neighbors, n_neighbors, num_rows, epochs, oversampler_name, undersampler_name, kind_sel, threshold_cleaning, n_seeds_condensed, allow_minority, version, n_neighbors_v3, voting_cluster, cv, kind, out_step):
    # Separar as amostras por classe
    X_majoritaria = X[y == 0]  # substitua 0 pela label da classe majoritária
    X_minoritaria = X[y == 1]  # substitua 1 pela label da classe minoritária


    y_majoritaria = y[y == 0]
    y_minoritaria = y[y == 1]

    state = 42
    sampling_strategy_down = 'majority'
    k_neighbors = k_neighbors
    n_neighbors = n_neighbors
    num_rows = num_rows
    epochs = epochs


    # Escolher técnicas individuais
    oversampler_name = oversampler_name
    undersampler_name = undersampler_name

    # Configurar downsampling
    if undersampler_name == "ENN":
      #ENN
      print("ENN") # Removed for cleaner output during optimization
      kind_sel_enn = kind_sel
      undersampler = EditedNearestNeighbours(n_neighbors=n_neighbors, kind_sel=kind_sel_enn, sampling_strategy = sampling_strategy_down, n_jobs = -1)

    elif undersampler_name == "Tomek":
      #Tomek
      print("Tomek") # Removed for cleaner output during optimization
      undersampler = TomekLinks(sampling_strategy= sampling_strategy_down, n_jobs = -1)


    elif undersampler_name == "Condensed Nearest Neighbour":
      #Condensed Nearest Neighbour
      print("Condensed Nearest Neighbour") # Removed for cleaner output during optimization
      n_seeds_condensed = n_seeds_condensed

      # Create Condensed Nearest Neighbour object with the suggested hyperparameters
      undersampler = CondensedNearestNeighbour(n_seeds_S = n_seeds_condensed, n_neighbors = n_neighbors, random_state = state, sampling_strategy= sampling_strategy_down) # Used state


    elif undersampler_name == "Neighbourhood Cleaning Rule":
      #Neighbourhood Cleaning Rule
      print("Neihbourhood Cleaning Rule") # Removed for cleaner output during optimization
      kind_sel_enn = kind_sel
      threshold_cleaning = threshold_cleaning
      kind_sel_neigh = kind_sel

      # Create Neighbourhood Cleaning Rule object with the suggested hyperparameters
      enn = EditedNearestNeighbours(n_neighbors=n_neighbors, kind_sel = kind_sel_enn, sampling_strategy = sampling_strategy_down, n_jobs = -1)
      undersampler = NeighbourhoodCleaningRule(edited_nearest_neighbours= enn, n_neighbors= n_neighbors, kind_sel = kind_sel_neigh, threshold_cleaning= threshold_cleaning, sampling_strategy= sampling_strategy_down, n_jobs = -1)


    elif undersampler_name == "Repeated Edited Nearest Neighbours":
      #Repeated Edited Nearest Neighbours
      print("Repeated Edited Nearest Neighbours") # Removed for cleaner output during optimization
      max_iter_repeated = max_iter_repeated
      kind_sel_repeated = kind_sel

      # Create Cluster Centroid object with the suggested hyperparameters
      undersampler = RepeatedEditedNearestNeighbours(max_iter = max_iter_repeated, kind_sel= kind_sel_repeated, n_neighbors = n_neighbors, sampling_strategy=sampling_strategy_down, n_jobs = -1)

    elif undersampler_name == "AllKNN":
      #AllKNN
      print("AllKNN") # Removed for cleaner output during optimization
      kind_sel_allknn = kind_sel
      allow_minority = allow_minority

      # Create AllKNN object with the suggested hyperparameters
      undersampler = AllKNN(allow_minority = allow_minority, kind_sel= kind_sel_allknn, n_neighbors = n_neighbors, sampling_strategy=sampling_strategy_down, n_jobs = -1)

    elif undersampler_name == "One Sided Selection":
      #One Sided Selection
      print("OneSided Selection")

      # Create One Sided Selection object with the suggested hyperparameters
      undersampler = OneSidedSelection(random_state = state, n_neighbors= n_neighbors, sampling_strategy= sampling_strategy_down, n_jobs = -1) # Used state

    elif undersampler_name == "NearMiss":
      #Nearmiss
      print("NearMiss")
      version = version
      n_neighbors_v3 = n_neighbors_v3

      # Create Cluster Centroid object with the suggested hyperparameters
      undersampler = NearMiss(version = version, n_neighbors= n_neighbors, n_neighbors_ver3= n_neighbors_v3, sampling_strategy=sampling_strategy_down, n_jobs = -1) # Changed sampling_strategy_up to sampling_strategy_down


    elif undersampler_name == "Cluster Centroids":
      #Cluster Centroids
      print("Cluster Centroids")
      voting_cluster = voting_cluster

      # Create Cluster Centroid object with the suggested hyperparameters7
      undersampler = ClusterCentroids(random_state = state, voting = voting_cluster, sampling_strategy=sampling_strategy_down) # Changed sampling_strategy_up to sampling_strategy_down


    elif undersampler_name == "Instance Hardness Threshold":
      #Instance Hardness Threshold
      print("Instance Hardness Threshold")
      cv = cv

      undersampler = InstanceHardnessThreshold(random_state = state, cv = cv, sampling_strategy=sampling_strategy_down, n_jobs = -1) # Used state


    # Aplicação do downsampling
    X_temp, y_temp = undersampler.fit_resample(X, y)
    samples_0 = len(y_temp[y_temp == 0])
    samples_1 = len(y_temp[y_temp == 1])
    samples = max(samples_0, samples_1)
    sampling_strategy_dict = { 1: samples, 0: samples}

    # Configurar upsampling
    if oversampler_name == 'SMOTE':
      print("SMOTE") # Removed for cleaner output during optimization
      oversampler = SMOTE(k_neighbors=k_neighbors, sampling_strategy=sampling_strategy_dict, random_state=state) # Added random_state

    elif oversampler_name == 'ADASYN':
      print("ADASYN") # Removed for cleaner output during optimization
      oversampler = ADASYN(random_state = state, n_neighbors = n_neighbors, sampling_strategy=sampling_strategy_dict) # Used state

    elif oversampler_name == 'Borderline SMOTE':
      print("Borderline SMOTE") # Removed for cleaner output during optimization
      kind = kind

      # Create Borderline SMOTE object with the suggested hyperparameters
      oversampler = BorderlineSMOTE(random_state = state, k_neighbors = k_neighbors, m_neighbors = n_neighbors, kind = kind, sampling_strategy=sampling_strategy_dict) # Used state

    elif oversampler_name == 'SVMSMOTE':
      print("SVMSMOTE") # Removed for cleaner output during optimization
      out_step = out_step

      # Create SVMSMOTE object with the suggested hyperparameters
      oversampler = SVMSMOTE(random_state = state, k_neighbors = k_neighbors, m_neighbors= n_neighbors, out_step = out_step, sampling_strategy=sampling_strategy_dict) # Used state


    X_resampled, y_resampled = oversampler.fit_resample(X_temp, y_temp)

    # Embaralhar o conjunto balanceado
    X_balanceado, y_balanceado = shuffle(X_resampled, y_resampled, random_state=42)

    return X_balanceado, y_balanceado



# 3. Pipeline Principal
def create_model(trial):
    # Classificador
    model_name = trial.suggest_categorical('classifier', ['XGBoost', 'LGBM', 'HistGradient', 'IF', 'RF', 'LOF', 'AdaBoostClassifier'])


    if model_name == 'XGBoost':
        model = XGBClassifier(
            n_estimators=trial.suggest_int('xgb_n_estimators', 50, 1000),
            max_leaves=trial.suggest_int('xgb_max_leaves', 0, 1000),
            max_bin=trial.suggest_int('xgb_max_bin', 10, 255),
            max_depth=trial.suggest_int('xgb_max_depth', 2, 100),
            learning_rate=trial.suggest_float('xgb_lr', 1e-4, 1.0, log=True),
            scale_pos_weight=trial.suggest_float('scale_pos_weight', 10, 2000),
            booster=trial.suggest_categorical('xgb_booster', ['gbtree', 'gblinear', 'dart']),
            eval_metric=trial.suggest_categorical('xgb_eval_metric', ['logloss', 'aucpr', 'auc', 'error']),
            use_label_encoder=False,
            tree_method='auto',
            n_jobs=-1
        )

    elif model_name == 'LGBM':
        model = LGBMClassifier(
            num_leaves=trial.suggest_int('lgbm_num_leaves', 15, 1000),  # Aumentar complexidade
            min_child_samples=trial.suggest_int('lgbm_min_child_samples', 1, 100),  # Permitir folhas menores
            min_split_gain=trial.suggest_float('lgbm_min_split_gain', 0.0, 0.5),  # Permitir ganhos menores
            reg_alpha=trial.suggest_float('lgbm_reg_alpha', 0, 1.0),
            reg_lambda=trial.suggest_float('lgbm_reg_lambda', 0, 1.0),
            n_estimators=trial.suggest_int('lgbm_n_estimators', 50, 1000),
            max_depth=trial.suggest_int('lgbm_max_depth', 2, 100),
            learning_rate=trial.suggest_float('lgbm_lr', 1e-4, 1.0, log=True),
            class_weight=trial.suggest_categorical('class_weight', ['balanced', None]),
            device='cpu',
            verbosity=-1
        )

    elif model_name == 'IF':
        model = IsolationForest(
            n_estimators=trial.suggest_int('if_n_estimators', 50, 1000),
            n_jobs=-1,
            random_state=42
        )

    elif model_name == 'HistGradient':
        model = HistGradientBoostingClassifier(
            max_leaf_nodes=trial.suggest_int('hgb_max_leaf_nodes', 10, 1000),
            min_samples_leaf=trial.suggest_int('hgb_min_samples_leaf', 10, 1000),
            max_bins=trial.suggest_int('hgb_max_bins', 10, 255),
            max_iter=trial.suggest_int('hgb_max_iter', 100, 1000),
            max_depth=trial.suggest_int('hgb_max_depth', 2, 100),
            learning_rate=trial.suggest_float('hgb_lr', 1e-4, 1.0, log=True),
            l2_regularization=trial.suggest_float('hgb_l2', 0.1, 10.0),
            class_weight=trial.suggest_categorical('hgb_class_weight', ['balanced', None])
        )

    elif model_name == 'RF':
      bootstrap = trial.suggest_categorical("bootstrap", [True, False])
      if bootstrap:
        max_samples = trial.suggest_int("max_samples", 2, 200)
        oob_score = trial.suggest_categorical("oob_score", [True, False])

      else:
        max_samples = None
        oob_score = False

      model = RandomForestClassifier(n_estimators = trial.suggest_int("n_estimators", 10, 350, log=True),
                                   criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
                                   max_depth = trial.suggest_int("max_depth", 1, 30),
                                   min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
                                   min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10),
                                   min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.01, 0.5),
                                   max_features = trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
                                   max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 200),
                                   bootstrap = bootstrap,
                                   oob_score = oob_score,
                                   n_jobs = -1,
                                   random_state = 42,
                                   verbose = 0,
                                   warm_start = trial.suggest_categorical("warm_start", [True, False]),
                                   class_weight = trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
                                   ccp_alpha = trial.suggest_float("ccp_alpha", 0.01, 1.0),
                                   max_samples = max_samples)


    elif model_name == 'LOF':
      model = LocalOutlierFactor(n_neighbors = trial.suggest_int("n_neighbors", 1, 30),
                                 algorithm = 'auto',
                                 leaf_size = trial.suggest_int("leaf_size", 1, 50),
                                 metric = 'minkowski',
                                 p = trial.suggest_int("p", 1, 5),
                                 contamination = 0.001665,
                                 novelty = True,
                                 n_jobs = -1)


    elif model_name == 'AdaBoostClassifier':
      model = AdaBoostClassifier(n_estimators = trial.suggest_int("n_estimators", 10, 350, log=True),
                                 learning_rate = trial.suggest_float("learning_rate", 0.01, 1),
                                 algorithm = 'SAMME',
                                 random_state = 42)


    return model

# FUNÇÃO PARA OTIMIZAÇÃO DE ENSEMBLE
def ensemble_objective(trial, base_models, X_train_processed, y_train_processed):
    """Função objetivo para otimização de estratégias de ensemble."""

    # Sugerir tipo de ensemble
    ensemble_type = trial.suggest_categorical('ensemble_type', [
        'Voting', 'Stacking', 'Bagging', 'AdaBoost', 'GradientBoosting'
    ])

    # Parâmetros comuns
    n_estimators = trial.suggest_int('ensemble_n_estimators', 10, 500)
    learning_rate = trial.suggest_float('ensemble_learning_rate', 0.0001, 1.0, log=True)


    if ensemble_type == 'Voting':
        # Configurar Voting Classifier
        voting = trial.suggest_categorical('voting_type', ['hard', 'soft'])
        weights = [trial.suggest_float(f'weight_{i}', 0.1, 1.0) for i in range(len(base_models))]
        ensemble = VotingClassifier(
            estimators=base_models,
            voting=voting,
            weights=weights,
            n_jobs=-1
        )


    elif ensemble_type == 'Stacking':
        # Configurar Stacking Classifier
        meta_classifier_name = trial.suggest_categorical('meta_classifier', [
            'LogisticRegression', 'RandomForest', 'XGBoost', 'LGBM'
        ])


        if meta_classifier_name == 'LogisticRegression':
            meta_classifier = LogisticRegression(
                C=trial.suggest_float('meta_C', 0.1, 10.0, log=True),
                max_iter=1000,
                n_jobs=-1
            )

        elif meta_classifier_name == 'RandomForest':
            meta_classifier = RandomForestClassifier(
                n_estimators=trial.suggest_int('meta_n_estimators', 20, 200),
                max_depth=trial.suggest_int('meta_max_depth', 2, 100),
                n_jobs=-1
            )

        elif meta_classifier_name == 'XGBoost':
            meta_classifier = XGBClassifier(
                n_estimators=trial.suggest_int('meta_n_estimators', 50, 200),
                max_depth=trial.suggest_int('meta_max_depth', 2, 100),
                learning_rate=trial.suggest_float('meta_learning_rate', 0.01, 0.3),
                n_jobs=-1
            )

        else:  # LGBM
            meta_classifier = LGBMClassifier(
                num_leaves=trial.suggest_int('meta_num_leaves', 15, 1000),  # Aumentar complexidade
                min_child_samples=trial.suggest_int('meta_min_child_samples', 1, 100),  # Permitir folhas menores
                min_split_gain=trial.suggest_float('meta_min_split_gain', 0.0, 0.5),  # Permitir ganhos menores
                reg_alpha=trial.suggest_float('meta_reg_alpha', 0, 1.0),
                reg_lambda=trial.suggest_float('meta_reg_lambda', 0, 1.0),
                n_estimators=trial.suggest_int('meta_n_estimators', 50, 1000),
                max_depth=trial.suggest_int('meta_max_depth', 2, 100),
                learning_rate=trial.suggest_float('meta_learning_rate', 1e-4, 1.0, log=True),
                class_weight=trial.suggest_categorical('meta_class_weight', ['balanced', None]),
                device='cpu',
                verbosity=-1
            )


        ensemble = StackingClassifier(
            estimators=base_models,
            final_estimator=meta_classifier,
            stack_method='predict_proba',
            n_jobs=-1
        )


    elif ensemble_type == 'Bagging':
        # Configurar Bagging Classifier
        max_samples = trial.suggest_float('bagging_max_samples', 0.05, 1.0)
        max_features = trial.suggest_float('bagging_max_features', 0.05, 1.0)


        ensemble = BaggingClassifier(
            estimator=base_models[0][1],
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=max_features,
            n_jobs=-1,
            random_state=42
        )



    elif ensemble_type == 'AdaBoost':
        # Configurar AdaBoost
        ensemble = AdaBoostClassifier(
            estimator=base_models[0][1],
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=42
        )



    else:  # GradientBoosting
        # Configurar Gradient Boosting
        max_depth = trial.suggest_int('gb_max_depth', 3, 10)
        subsample = trial.suggest_float('gb_subsample', 0.5, 1.0)
        min_samples_split = trial.suggest_int('gb_min_samples_split', 2, 20)


        ensemble = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            min_samples_split=min_samples_split,
            random_state=42
        )



    # Avaliação com validação cruzada
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []


    for train_idx, val_idx in cv.split(X_train_processed, y_train_processed):
        X_train_fold, X_val_fold = X_train_processed[train_idx], X_train_processed[val_idx]
        y_train_fold, y_val_fold = y_train_processed[train_idx], y_train_processed[val_idx]


        try:
            ensemble.fit(X_train_fold, y_train_fold)
            y_pred = ensemble.predict(X_val_fold)
            score = f1_score(y_val_fold, y_pred, average='macro')
            scores.append(score)

        except Exception as e:
            print(f"Erro no fold: {str(e)}; ensemble_objective")
            scores.append(0.0)

    return np.mean(scores)



# 4. Função de Otimização simplificada
def objective(trial):
    rm_outliers = trial.suggest_categorical('rm_outliers', [True, False])
    apply_fw = trial.suggest_categorical('apply_fw', [True, False])

    # Parâmetros
    if(rm_outliers):
        outlier_params = {
            'iqr_multiplier': trial.suggest_float('outlier_multiplier', 1.0, 5.0),
            'strategy': trial.suggest_categorical('outlier_strategy', ['cap', 'remove']),
            'quantil': trial.suggest_float('quantil', 0.05, 0.25)
        }


    if(apply_fw):
        fw_params = {
            'corr_limit': trial.suggest_float('fw_corr_limit', 0.4, 0.99),
            'feature_engg': trial.suggest_categorical('fw_engg', ['', 'groupby', 'target']), #'interactions' retirado -> estava gerando problemas com o dask
            'category_encoders': trial.suggest_categorical('fw_encoders', ['', 'target', 'auto']),
            'auto_encoders': trial.suggest_categorical('fw_auto_encoders', ['', 'dae', 'dae_add', 'vae', 'vae_add', 'gan']), #'cnn' e 'cnn_add' removidos para resolução de conflito: Erro no Featurewiz: The total size of the tensor must be unchanged, however, the input size cannot by divided by the specified dimensions in target_shape. Received: input_shape=(30,), target_shape=(7, -1)
            'imbalanced': trial.suggest_categorical('fw_imbalanced', [True, False])
        }


    # ADICIONADO NOVA ESTRATÉGIA 'COMBO'
    balance_strategy = trial.suggest_categorical('balance_strategy', [
        'SMOTE', 'ADASYN', 'SMOTE+TL', 'SMOTE-ENN',
        'Borderline-SMOTE', 'KMeans-SMOTE',
        'IHT', 'NearMiss', 'TL', 'Combo', 'Balanced'])

    # Parâmetros comuns
    sampling_strategy = trial.suggest_float('sampling_strategy', 0.025, 0.975)
    k_neighbors = trial.suggest_int('k_neighbors', 2, 20)
    nearMissVersion = trial.suggest_int('nearMissVersion', 1, 3)
    downsampling_strategy = trial.suggest_float('downsampling_strategy', 0.025, 0.975)
    n_neighbors_nm = trial.suggest_int('n_neighbors_nm', 2, 20)
    n_neighbors_ver3_nm = trial.suggest_int('n_neighbors_ver3_nm', 2, 20)

    # Escolher técnicas individuais
    oversampler_name = trial.suggest_categorical('combo_oversampler',
                                               ['SMOTE', 'ADASYN', 'BorderlineSMOTE', 'KMeansSMOTE', 'SVMSMOTE'])

    undersampler_name = trial.suggest_categorical('combo_undersampler',
                                                 ['TomekLinks', 'IHT', 'NearMiss', 'ENN', 'Condensed Nearest Neighbour',
                                                  'Neighbourhood Cleaning Rule', 'Repeated Edited Nearest Neighbours',
                                                  'AllKNN', 'One Sided Selection', 'Cluster Centroids'])

    print(f"Tamanho inicial antes do loop: X_train={X_train.shape}, y_train={y_train.shape}")

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
        X_train_fold, X_valid_fold = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_train_fold, y_valid_fold = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        try:
            # 1. Remoção de outliers
            if(rm_outliers):
                X_train_fold, y_train_fold = remove_outliers(
                    X_train_fold, y_train_fold,
                    iqr_multiplier=outlier_params['iqr_multiplier'],
                    strategy=outlier_params['strategy'],
                    quantil=outlier_params['quantil']
                )


            # Verificar se restaram amostras suficientes
            if len(X_train_fold) < 10 or y_train_fold.nunique() < 2:
                print(f"Fold {fold}: Amostras insuficientes após remoção de outliers")
                continue


            # 2. Featurewiz
            if(apply_fw):
                X_train_fw, X_valid_fw, y_train_fold, _ = apply_featurewiz(
                    X_train_fold, y_train_fold, X_valid_fold, **fw_params
                )

            else:
                X_train_fw = X_train_fold
                X_valid_fw = X_valid_fold


            # 3. Balanceamento
            if balance_strategy == 'Balanced':
                k_neighbors = trial.suggest_int("k_neighbours", 3, 10)
                n_neighbors = trial.suggest_int("n_neighbors", 3, 10)
                num_rows = trial.suggest_int('num_rows', 1, 10000)
                epochs = trial.suggest_int('epochs', 50, 300)


                # Escolher técnicas individuais
                oversampler_name = trial.suggest_categorical('oversampler',
                                                            ["SMOTE", "ADASYN", "Borderline SMOTE", "SVMSMOTE"])

                undersampler_name = trial.suggest_categorical('undersampler',
                                                             ["ENN", "Tomek", "Condensed Nearest Neighbour",
                                                              "Neighbourhood Cleaning Rule", "Repeated Edited Nearest Neighbours",
                                                              "AllKNN", "One Sided Selection", "NearMiss", "Cluster Centroids",
                                                              "Instance Hardness Threshold"])

                kind_sel = trial.suggest_categorical("kind_sel", ["all", "mode"])
                threshold_cleaning = trial.suggest_float('threshold_cleaning', 0.1, 1.0)
                n_seeds_condensed = trial.suggest_int('n_seeds_condensed', 1, 10)
                allow_minority = trial.suggest_categorical('allow_minority', [True, False])
                version = trial.suggest_int('version', 1,3)
                n_neighbors_v3 = trial.suggest_int('n_neighbors_v3', 3, 10)
                voting_cluster = trial.suggest_categorical('voting_cluster', ['hard', 'soft', 'auto'])
                cv = trial.suggest_int('cv', 2, 50)
                kind = trial.suggest_categorical('kind', ['borderline-1', 'borderline-2'])
                out_step = trial.suggest_float('out_step', 0.1, 1.0)

                X_train_bal, y_train_bal = balance_dataset(X_train_fw, y_train_fold, k_neighbors, n_neighbors, num_rows, epochs, oversampler_name, undersampler_name, kind_sel, threshold_cleaning, n_seeds_condensed, allow_minority, version, n_neighbors_v3, voting_cluster, cv, kind, out_step)

            elif balance_strategy == 'SMOTE':
                sampler = SMOTE(
                    k_neighbors=k_neighbors,
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'ADASYN':
                sampler = ADASYN(
                    n_neighbors=k_neighbors,
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'SMOTE+TL':
                smote = SMOTE(
                    k_neighbors=k_neighbors,
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                tl = TomekLinks(n_jobs=-1)
                X_train_smote, y_train_smote = smote.fit_resample(X_train_fw, y_train_fold)
                X_train_bal, y_train_bal = tl.fit_resample(X_train_smote, y_train_smote)


            elif balance_strategy == 'TL':
                sampler = TomekLinks(n_jobs=-1)
                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'SMOTE-ENN':
                sampler = SMOTEENN(
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'Borderline-SMOTE':
                sampler = BorderlineSMOTE(
                    k_neighbors=k_neighbors,
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'KMeans-SMOTE':
                sampler = KMeansSMOTE(
                    k_neighbors=k_neighbors,
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'IHT':  # Instance Hardness Threshold
                base_estimator = LogisticRegression(
                    class_weight='balanced',
                    max_iter=1000,
                    random_state=42
                )

                sampler = InstanceHardnessThreshold(
                    estimator=base_estimator,
                    sampling_strategy=sampling_strategy,
                    random_state=42
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)


            elif balance_strategy == 'NearMiss':
                sampler = NearMiss(
                    version=nearMissVersion,
                    sampling_strategy=sampling_strategy
                )

                X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_fold)# NOVA ESTRATÉGIA: COMBINAÇÃO DE UPSAMPLING E DOWNSAMPLING


            elif balance_strategy == 'Combo':
                sequence = trial.suggest_categorical('combo_sequence', ['UpDown', 'DownUp'])
                sampling_strategy_down = 'majority'

                # Configurar upsampling
                if oversampler_name == 'SMOTE':
                    oversampler = SMOTE(
                        k_neighbors=k_neighbors,
                        sampling_strategy=sampling_strategy,
                        random_state=42
                    )

                elif oversampler_name == 'ADASYN':
                    oversampler = ADASYN(
                        n_neighbors=k_neighbors,
                        sampling_strategy=sampling_strategy,
                        random_state=42
                    )

                elif oversampler_name == 'BorderlineSMOTE':
                    oversampler = BorderlineSMOTE(
                        k_neighbors=k_neighbors,
                        sampling_strategy=sampling_strategy,
                        random_state=42
                    )

                elif oversampler_name == 'KMeansSMOTE':
                    oversampler = KMeansSMOTE(
                        k_neighbors=k_neighbors,
                        sampling_strategy=sampling_strategy,
                        random_state=42
                    )

                elif oversampler_name == 'SVMSMOTE':
                    oversampler = SVMSMOTE(
                        k_neighbors=k_neighbors,
                        sampling_strategy=sampling_strategy,
                        random_state=42
                    )


                # Configurar downsampling
                if undersampler_name == 'TomekLinks':
                    undersampler = TomekLinks(n_jobs=-1)

                elif undersampler_name == 'IHT':
                    base_estimator = LogisticRegression(
                        class_weight='balanced',
                        max_iter=1000,
                        random_state=42
                    )

                    undersampler = InstanceHardnessThreshold(
                        estimator=base_estimator,
                        sampling_strategy=downsampling_strategy,
                        random_state=42,
                        n_jobs=-1
                    )

                elif undersampler_name == 'NearMiss':
                    undersampler = NearMiss(
                        version=nearMissVersion,
                        sampling_strategy=downsampling_strategy,
                        n_neighbors=n_neighbors_nm,
                        n_neighbors_ver3=n_neighbors_ver3_nm,
                        n_jobs=-1
                    )

                elif undersampler_name == 'ENN':
                    undersampler = EditedNearestNeighbours(
                        sampling_strategy=sampling_strategy_down,
                        n_neighbors=n_neighbors_nm,
                        n_jobs=-1
                    )

                elif undersampler_name == 'Condensed Nearest Neighbour':
                    undersampler = CondensedNearestNeighbour(
                        sampling_strategy=sampling_strategy_down,
                        n_jobs=-1
                    )

                elif undersampler_name == 'Neighbourhood Cleaning Rule':
                    undersampler = NeighbourhoodCleaningRule(
                        sampling_strategy=sampling_strategy_down,
                        n_jobs=-1
                    )

                elif undersampler_name == 'Repeated Edited Nearest Neighbours':
                    undersampler = RepeatedEditedNearestNeighbours(
                        sampling_strategy=sampling_strategy_down,
                        n_jobs=-1
                    )

                elif undersampler_name == 'AllKNN':
                    undersampler = AllKNN(
                        sampling_strategy=sampling_strategy_down,
                        n_jobs=-1
                    )

                elif undersampler_name == 'One Sided Selection':
                    undersampler = OneSidedSelection(
                        sampling_strategy=sampling_strategy_down,
                        n_jobs=-1
                    )

                elif undersampler_name == 'Cluster Centroids':
                    undersampler = ClusterCentroids(
                        sampling_strategy=downsampling_strategy,
                        n_jobs=-1
                    )



                # Aplicação sequencial
                if sequence == 'UpDown':
                    X_temp, y_temp = oversampler.fit_resample(X_train_fw, y_train_fold)
                    X_train_bal, y_train_bal = undersampler.fit_resample(X_temp, y_temp)


                elif sequence == 'DownUp':
                    X_temp, y_temp = undersampler.fit_resample(X_train_fw, y_train_fold)
                    X_train_bal, y_train_bal = oversampler.fit_resample(X_temp, y_temp)


            print(f"Distribuição após balanceamento: {np.bincount(y_train_bal)}")

            if len(np.unique(y_train_bal)) < 2:
                raise ValueError("Dataset perdeu classes após balanceamento!")


            # 4. Padronização
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_bal)
            X_valid_scaled = scaler.transform(X_valid_fw)


            # 5. Modelagem
            model = create_model(trial)
            model.fit(X_train_scaled, y_train_bal)


            # 6. Avaliação
            y_pred = model.predict(X_valid_scaled)


            # Garantir que temos previsões para as classes
            if 0 not in np.unique(y_pred):
                print(f"Fold {fold}: Classe majoritária não prevista")
                score = 0.0

            elif 1 not in np.unique(y_pred):
                print(f"Fold {fold}: Classe minoritária não prevista")
                score = 0.0

            else:
                score = f1_score(y_valid_fold, y_pred, average='macro')


            scores.append(score)


        except Exception as e:
            print(f"Erro no fold {fold}: {str(e)}; objective")
            scores.append(0.0)


    # Retornar média dos scores válidos
    valid_scores = [s for s in scores if s > 0]
    return np.mean(valid_scores) if valid_scores else 0.0



# 5. Execução Principal
if __name__ == "__main__":

    # Carregar dados
    data = pd.read_csv(r"C:\Users\lanna\Downloads\creditcardfraud\creditcard.csv")
    X = data.drop(['Class', 'Time'], axis=1)
    y = data['Class'].astype(int)

    # Normalização da coluna Amount com RobustScaler() -> mais imune e trabalha bem em casos com outliers
    amount = pd.DataFrame(X["Amount"])
    train_scaled = RobustScaler().fit(amount)
    X["Amount"] = train_scaled.transform(amount)


    # Divisão estratificada
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )


    print(f"Distribuição original:\n{y.value_counts()}")
    print(f"Distribuição treino:\n{y_train.value_counts()}")


    # Estudo Optuna
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=13),
        pruner=optuna.pruners.HyperbandPruner(),
        study_name='credit_card_ensemblev11',
        storage="sqlite:///credit_card_ensamble_storage.db",
        load_if_exists=True
    )


    # Otimização
    study.optimize(objective, n_trials=50)


    print(f"Melhor trial: Valor F1 = {study.best_value:.4f}")
    print("Melhores parâmetros:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")


    # Processamento final com melhores parâmetros
    # 1. Remoção de outliers
    rm_outliers = study.best_params.get('rm_outliers', False)
    if(rm_outliers):
        X_train_final, y_train_final = remove_outliers(
            X_train, y_train,
            iqr_multiplier=study.best_params.get('outlier_multiplier', 3.0),
            strategy=study.best_params.get('outlier_strategy', 'cap'),
            quantil=study.best_params.get('quantil', 0.25)
        )

    else:
        X_train_final = X_train
        y_train_final = y_train


    print(f"Amostras após remoção de outliers: {len(X_train_final)}")


    # 2. Featurewiz
    apply_fw = study.best_params.get('apply_fw', False)
    if(apply_fw):
        fw_params = {
            'corr_limit': study.best_params.get('fw_corr_limit', 0.7),
            'feature_engg': study.best_params.get('fw_engg', ''),
            'category_encoders': study.best_params.get('fw_encoders', ''),
            'auto_encoders': study.best_params.get('fw_auto_encoders', ''),
            'imbalanced': study.best_params.get('fw_imbalanced', False)
        }

        X_train_fw, X_test_fw, y_train_final ,features = apply_featurewiz(
            X_train_final, y_train_final, X_test, **fw_params
        )

    else:
        X_train_fw = X_train_final
        X_test_fw = X_test
        features = np.ones((X_train_fw.shape[1],1))


    # 3. Balanceamento - ADICIONADA LÓGICA PARA 'COMBO'
    balance_strategy = study.best_params.get('balance_strategy', 'SMOTE')
    sampling_strategy = study.best_params.get('sampling_strategy', 0.5)
    k_neighbors = study.best_params.get('k_neighbors', 5)


    downsampling_strategy = study.best_params.get('downsampling_strategy', 'auto')
    n_neighbors_nm = study.best_params.get('n_neighbors_nm', 3)
    n_neighbors_ver3_nm = study.best_params.get('n_neighbors_ver3_nm', 3)


    # Recuperar técnicas escolhidas
    oversampler_name = study.best_params.get('combo_oversampler', 'SMOTE')
    undersampler_name = study.best_params.get('combo_undersampler', 'TomekLinks')


    if balance_strategy == 'Balanced':
        k_neighbors = study.best_params.get("k_neighbours", 3)
        n_neighbors = study.best_params.get("n_neighbors", 3)
        num_rows = study.best_params.get('num_rows', 1000)
        epochs = study.best_params.get('epochs', 100)
        oversampler_name = study.best_params.get('combo_oversampler', 'SMOTE')
        undersampler_name = study.best_params.get('combo_undersampler', 'Tomek')
        kind_sel = study.best_params.get("kind_sel", "all")
        threshold_cleaning = study.best_params.get('threshold_cleaning', 0.5)
        n_seeds_condensed = study.best_params.get('n_seeds_condensed', 5)
        allow_minority = study.best_params.get('allow_minority', True)
        version = study.best_params.get('version', 1)
        n_neighbors_v3 = study.best_params.get('n_neighbors_v3', 3)
        voting_cluster = study.best_params.get('voting_cluster', 'hard')
        cv = study.best_params.get('cv', 5)
        kind = study.best_params.get('kind', 'borderline-1')
        out_step = study.best_params.get('out_step', 0.5)

        X_train_bal, y_train_bal = balance_dataset(X_train_fw, y_train_final, k_neighbors, n_neighbors, num_rows, epochs, oversampler_name, undersampler_name, kind_sel, threshold_cleaning, n_seeds_condensed, allow_minority, version, n_neighbors_v3, voting_cluster, cv, kind, out_step)


    elif balance_strategy == 'SMOTE':
        sampler = SMOTE(
            k_neighbors=k_neighbors,
            sampling_strategy=sampling_strategy,
            random_state=42
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'ADASYN':
        sampler = ADASYN(
            n_neighbors=k_neighbors,
            sampling_strategy=sampling_strategy,
            random_state=42
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'SMOTE+TL':
        smote = SMOTE(
            k_neighbors=k_neighbors,
            sampling_strategy=sampling_strategy,
            random_state=42
        )

        tl = TomekLinks(n_jobs=-1)
        X_train_smote, y_train_smote = smote.fit_resample(X_train_fw, y_train_final)
        X_train_bal, y_train_bal = tl.fit_resample(X_train_smote, y_train_smote)


    elif balance_strategy == 'TL':
        sampler = TomekLinks(n_jobs=-1)
        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'SMOTE-ENN':
        sampler = SMOTEENN(
            sampling_strategy=sampling_strategy,
            random_state=42
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'Borderline-SMOTE':
        sampler = BorderlineSMOTE(
            k_neighbors=k_neighbors,
            sampling_strategy=sampling_strategy,
            random_state=42
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'KMeans-SMOTE':
        sampler = KMeansSMOTE(
            k_neighbors=k_neighbors,
            sampling_strategy=sampling_strategy,
            random_state=42
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'IHT':
        base_estimator = LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        )

        sampler = InstanceHardnessThreshold(
            estimator=base_estimator,
            sampling_strategy=downsampling_strategy,
            random_state=42,
            n_jobs=-1
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    elif balance_strategy == 'NearMiss':
        sampler = NearMiss(
            version=n_neighbors_ver3_nm,
            sampling_strategy=downsampling_strategy,
            n_neighbors=n_neighbors_nm,
            n_neighbors_ver3=n_neighbors_ver3_nm,
            n_jobs=-1
        )

        X_train_bal, y_train_bal = sampler.fit_resample(X_train_fw, y_train_final)


    # NOVA ESTRATÉGIA: COMBINAÇÃO DE UPSAMPLING E DOWNSAMPLING
    elif balance_strategy == 'Combo':
        version = study.best_params.get('version', 1)
        sequence = study.best_params.get('combo_sequence', 'UpDown')

        # Configurar upsampling
        if oversampler_name == 'SMOTE':
            oversampler = SMOTE(
                k_neighbors=k_neighbors,
                sampling_strategy=sampling_strategy,
                random_state=42
            )

        elif oversampler_name == 'ADASYN':
            oversampler = ADASYN(
                n_neighbors=k_neighbors,
                sampling_strategy=sampling_strategy,
                random_state=42
            )

        elif oversampler_name == 'BorderlineSMOTE':
            oversampler = BorderlineSMOTE(
                k_neighbors=k_neighbors,
                sampling_strategy=sampling_strategy,
                random_state=42
            )

        elif oversampler_name == 'KMeansSMOTE':
            oversampler = KMeansSMOTE(
                k_neighbors=k_neighbors,
                sampling_strategy=sampling_strategy,
                random_state=42
            )

        elif oversampler_name == 'SVMSMOTE':
                    oversampler = SVMSMOTE(
                        k_neighbors=k_neighbors,
                        sampling_strategy=sampling_strategy,
                        random_state=42
                    )


        # Configurar downsampling
        if undersampler_name == 'TomekLinks':
            undersampler = TomekLinks(n_jobs=-1)

        elif undersampler_name == 'IHT':
            base_estimator = LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=42
            )

            undersampler = InstanceHardnessThreshold(
                estimator=base_estimator,
                sampling_strategy=downsampling_strategy,
                random_state=42,
                n_jobs=-1
            )

        elif undersampler_name == 'NearMiss':
            undersampler = NearMiss(
                version= version,
                sampling_strategy=downsampling_strategy,
                n_neighbors=n_neighbors_nm,
                n_neighbors_ver3=n_neighbors_ver3_nm,
                n_jobs=-1
            )

        elif undersampler_name == 'ENN':
            undersampler = EditedNearestNeighbours(
                sampling_strategy=downsampling_strategy,
                n_neighbors=n_neighbors_nm,
                n_jobs=-1
            )

        elif undersampler_name == 'Condensed Nearest Neighbour':
            undersampler = CondensedNearestNeighbour(
                sampling_strategy=downsampling_strategy,
                n_jobs=-1
            )

        elif undersampler_name == 'Neighbourhood Cleaning Rule':
            undersampler = NeighbourhoodCleaningRule(
                sampling_strategy=downsampling_strategy,
                n_jobs=-1
            )

        elif undersampler_name == 'Repeated Edited Nearest Neighbours':
            undersampler = RepeatedEditedNearestNeighbours(
                sampling_strategy=downsampling_strategy,
                n_jobs=-1
            )

        elif undersampler_name == 'AllKNN':
            undersampler = AllKNN(
                sampling_strategy=downsampling_strategy,
                n_jobs=-1
            )

        elif undersampler_name == 'One Sided Selection':
            undersampler = OneSidedSelection(
                sampling_strategy=downsampling_strategy,
                n_jobs=-1
            )

        elif undersampler_name == 'Cluster Centroids':
            undersampler = ClusterCentroids(
                sampling_strategy=downsampling_strategy,
                n_jobs=-1
            )


        # Aplicação sequencial
        if sequence == 'UpDown':
            X_temp, y_temp = oversampler.fit_resample(X_train_fw, y_train_final)
            X_train_bal, y_train_bal = undersampler.fit_resample(X_temp, y_temp)


        elif sequence == 'DownUp':
            X_temp, y_temp = undersampler.fit_resample(X_train_fw, y_train_final)
            X_train_bal, y_train_bal = oversampler.fit_resample(X_temp, y_temp)


    print(f"Distribuição após balanceamento: {np.bincount(y_train_bal)}")

    if len(np.unique(y_train_bal)) < 2:
        raise ValueError("Dataset perdeu classes após balanceamento!")


    # 4. Padronização
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal)
    X_test_scaled = scaler.transform(X_test_fw)


    # 5. Modelagem
    model = create_model(study.best_trial)
    model.fit(X_train_scaled, y_train_bal)


    # 6. Avaliação
    # Start the timer
    start_time = time.time()

    #Start RAM
    # Get virtual memory statistics
    mem_info_start = psutil.virtual_memory()

    #predict
    y_pred = model.predict(X_test_scaled)

    # End the timer
    end_time = time.time()

    #End RAM
    # Get virtual memory statistics
    mem_info_end = psutil.virtual_memory()

    # Calculate elapsed time
    elapsed_time = end_time - start_time
    print(f"Tempo: {elapsed_time:.4f} segundos")

    #Calculate used RAM in prediction
    used_ram_pred = mem_info_end.used - mem_info_start.used
    print(f"RAM used in prediction: {used_ram_pred / (1024**3):.2f} GB")

    print("\nRelatório de Classificação do melhor modelo individual:")
    print(classification_report(y_test, y_pred, target_names=['Classe Negativa', 'Classe Positiva']))


    # Mapeia cada valor para o objeto Trial, lidando com valores duplicados
    # (mantém a primeira ocorrência do Trial para cada valor)
    value_to_trial = {t.value: t for t in study.trials if t.value is not None}

    # Extrai os valores únicos e ordena
    distinct_values = sorted(value_to_trial.keys(), reverse=True)

    # Seleciona os 10 maiores valores únicos
    top_10_distinct_values = distinct_values[:10]

    # Agora, você pode pegar os objetos Trial correspondentes e usá-los
    top_trials = [value_to_trial[v] for v in top_10_distinct_values]

    print("\nTop 10 trials selecionados para o ensemble:")
    for i, trial in enumerate(top_trials):
        print(f"Trial {i+1}: F1-score = {trial.value:.4f}")

    # 8. Criar modelos para o ensemble
    base_models  = []
    for i, trial in enumerate(top_trials):
        print(f"\nCriando modelo {i+1}/{len(top_trials)} para o ensemble...")


        # Criar e treinar o modelo
        model_i = create_model(trial)
        base_models.append((f'model_{i}', model_i))


    # NOVO ESTUDO PARA ENSEMBLES
    ensemble_study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        study_name='credit_card_ensemble_strategiesv11',
        storage="sqlite:///credit_card_ensemble_storage.db",
        load_if_exists=True
    )


    # Função objetivo parcial para passar argumentos
    def partial_objective(trial):
        return ensemble_objective(
            trial,
            base_models,
            X_train_scaled,
            y_train_bal
        )



    # Otimização de ensembles
    ensemble_study.optimize(partial_objective, n_trials=100)


    print(f"\nMelhor ensemble: F1-score = {ensemble_study.best_value:.4f}")
    print("Melhores parâmetros para ensemble:")
    for key, value in ensemble_study.best_params.items():
        print(f"  {key}: {value}")


    # Reconstruir o melhor ensemble
    best_ensemble_params = ensemble_study.best_params
    ensemble_type = best_ensemble_params['ensemble_type']


    if ensemble_type == 'Voting':
        weights = [best_ensemble_params[f'weight_{i}'] for i in range(len(base_models))]
        best_ensemble = VotingClassifier(
            estimators=base_models,
            voting=best_ensemble_params['voting_type'],
            weights=weights,
            n_jobs=-1
        )


    elif ensemble_type == 'Stacking':
        # Configurar meta-classificador
        meta_classifier_name = best_ensemble_params['meta_classifier']


        if meta_classifier_name == 'LogisticRegression':
            meta_classifier = LogisticRegression(
                C=best_ensemble_params.get('meta_C', 1.0),
                max_iter=1000,
                n_jobs=-1
            )

        elif meta_classifier_name == 'RandomForest':
            meta_classifier = RandomForestClassifier(
                n_estimators=best_ensemble_params.get('meta_n_estimators', 100),
                max_depth=best_ensemble_params.get('meta_max_depth', 5),
                n_jobs=-1
            )

        elif meta_classifier_name == 'XGBoost':
            meta_classifier = XGBClassifier(
                n_estimators=best_ensemble_params.get('meta_n_estimators', 100),
                max_depth=best_ensemble_params.get('meta_max_depth', 5),
                learning_rate=best_ensemble_params.get('meta_learning_rate', 0.1),
                n_jobs=-1
            )

        else:  # LGBM
            meta_classifier = LGBMClassifier(
                num_leaves=best_ensemble_params.get('meta_num_leaves', 15),  # Aumentar complexidade
                min_child_samples=best_ensemble_params.get('meta_min_child_samples', 1),  # Permitir folhas menores
                min_split_gain=best_ensemble_params.get('meta_min_split_gain', 0.0),  # Permitir ganhos menores
                reg_alpha=best_ensemble_params.get('meta_reg_alpha', 0),
                reg_lambda=best_ensemble_params.get('meta_reg_lambda', 0),
                n_estimators=best_ensemble_params.get('meta_n_estimators', 50),
                max_depth=best_ensemble_params.get('meta_max_depth', 2),
                learning_rate=best_ensemble_params.get('meta_learning_rate', 1e-4),
                class_weight=best_ensemble_params.get('meta_class_weight', None),
                device='cpu',
                verbosity=-1
            )


        best_ensemble = StackingClassifier(
            estimators=base_models,
            final_estimator=meta_classifier,
            stack_method='predict_proba',
            n_jobs=-1
        )


    elif ensemble_type == 'Bagging':
        best_ensemble = BaggingClassifier(
            estimator=base_models[0][1],
            n_estimators=best_ensemble_params['ensemble_n_estimators'],
            max_samples=best_ensemble_params['bagging_max_samples'],
            max_features=best_ensemble_params['bagging_max_features'],
            n_jobs=-1,
            random_state=42
        )


    elif ensemble_type == 'AdaBoost':
        best_ensemble = AdaBoostClassifier(
            estimator=base_models[0][1],
            n_estimators=best_ensemble_params['ensemble_n_estimators'],
            learning_rate=best_ensemble_params['ensemble_learning_rate'],
            random_state=42
        )


    else:  # GradientBoosting
        best_ensemble = GradientBoostingClassifier(
            n_estimators=best_ensemble_params['ensemble_n_estimators'],
            learning_rate=best_ensemble_params['ensemble_learning_rate'],
            max_depth=best_ensemble_params['gb_max_depth'],
            subsample=best_ensemble_params['gb_subsample'],
            min_samples_split=best_ensemble_params.get('gb_min_samples_split', 2),
            random_state=42
        )



    # Treinar o melhor ensemble
    best_ensemble.fit(X_train_scaled, y_train_bal)


    # Avaliar no conjunto de teste
    # Start the timer
    start_time = time.time()

    #Start RAM
    # Get virtual memory statistics
    mem_info_start = psutil.virtual_memory()

    #predict
    y_pred_ensemble = best_ensemble.predict(X_test_scaled)

    # End the timer
    end_time = time.time()

    #End RAM
    # Get virtual memory statistics
    mem_info_end = psutil.virtual_memory()

    # Calculate elapsed time
    elapsed_time = end_time - start_time
    print(f"Tempo: {elapsed_time:.4f} segundos")

    #Calculate used RAM in prediction
    used_ram_pred = mem_info_end.used - mem_info_start.used
    print(f"RAM used in prediction: {used_ram_pred / (1024**3):.2f} GB")

    print("\nRelatório de Classificação do melhor ensemble:")
    print(classification_report(y_test, y_pred_ensemble, target_names=['Classe Negativa', 'Classe Positiva']))


    # Salvar o melhor ensemble
    ensemble_data = {
        'ensemble': best_ensemble,
        'base_models': base_models,
        'type': ensemble_type,
        'params': best_ensemble_params,
        'scaler': scaler,
        'features': features,
        'f1_score': ensemble_study.best_value,
        'classification_report': classification_report(y_test, y_pred_ensemble, output_dict=True)
    }


    joblib.dump(ensemble_data, 'melhor_ensemble.pkl')
    print("Melhor ensemble salvo como 'melhor_ensemble.pkl'")


    # 12. Ajuste de Threshold
    if hasattr(best_ensemble, 'predict_proba'):
        y_probs = best_ensemble.predict_proba(X_test_scaled)


        thresholds = np.linspace(0, 1, 1000)
        f1_scores = [f1_score(y_test, (y_probs[:, 1] > t).astype(int), average='macro') for t in thresholds]
        best_threshold = thresholds[np.argmax(f1_scores)]

        print(f"Melhor threshold: {best_threshold:.4f}, F1-score: {max(f1_scores):.4f}")

    else:
        print("O classificador não suporta predict_proba. Pulando ajuste de threshold.")


    # 13. Feature Importance
    if hasattr(best_ensemble, 'feature_importances_'):

        try:
            importances = best_ensemble.feature_importances_
            fi_df = pd.DataFrame({'Feature': features, 'Importance': importances})
            fi_df = fi_df.sort_values('Importance', ascending=False)

            print("\nTop 20 Features por Importância:")
            print(fi_df.head(20))

        except Exception as e:
            print(f"Não foi possível obter importância de features: {str(e)}")


    # 14. Salvar modelo individual
    model_data = {
        'model': model,
        'scaler': scaler,
        'features': features,
        'f1_score': study.best_value,
        'best_params': study.best_params,
        'threshold': best_threshold if 'best_threshold' in locals() else 0.5
    }

    joblib.dump(model_data, 'melhor_modelo_individual.pkl')

    print("Modelo individual e ensembles salvos!")

[I 2025-08-23 16:17:32,617] Using an existing study with name 'credit_card_ensemblev5' instead of creating a new one.


Distribuição original:
Class
0    284315
1       492
Name: count, dtype: int64
Distribuição treino:
Class
0    227451
1       394
Name: count, dtype: int64
Tamanho inicial antes do loop: X_train=(227845, 29), y_train=(227845,)
Erro no fold 0: No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.
Erro no fold 1: No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.


[I 2025-08-23 16:17:34,369] Trial 6 finished with value: 0.0 and parameters: {'rm_outliers': True, 'apply_fw': False, 'outlier_multiplier': 4.890404455619573, 'outlier_strategy': 'remove', 'quantil': 0.20510530292096935, 'balance_strategy': 'KMeans-SMOTE', 'sampling_strategy': 0.36541709356972124, 'k_neighbors': 20, 'nearMissVersion': 1, 'downsampling_strategy': 0.32842179806134636, 'n_neighbors_nm': 19, 'n_neighbors_ver3_nm': 2, 'combo_oversampler': 'BorderlineSMOTE', 'combo_undersampler': 'Neighbourhood Cleaning Rule'}. Best is trial 0 with value: 0.0.


Erro no fold 2: No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.
Tamanho inicial antes do loop: X_train=(227845, 29), y_train=(227845,)
featurewiz is given 0.7 as correlation limit...
    moving target encoder from feature_engg to category_encoders list
    final list of feature engineering given: []
    final list of category encoders given: ['label', 'label', 'target']
Since Auto Encoders are selected for feature extraction,
    SULOV is skipped since auto-encoders is given...
    Recursive XGBoost is also skipped...
GANAugmenter(gan_model=<class 'featurewiz.auto_encoders.GAN'>)
    AE dictionary given: dict_items([])
    final list of scalers given: [minmax]
Loaded input data. Shape = (151896, 29)
#### Starting featurewiz transform for train data ####
    Single_Label Binary_Classification problem 
Shape of dataset: (151896, 29). Now we classify variables into different types...
Time taken to defin

[I 2025-08-24 00:08:49,700] Trial 7 finished with value: 0.0 and parameters: {'rm_outliers': False, 'apply_fw': True, 'fw_corr_limit': 0.6702706321932905, 'fw_engg': 'target', 'fw_encoders': '', 'fw_auto_encoders': 'gan', 'fw_imbalanced': True, 'balance_strategy': 'Combo', 'sampling_strategy': 0.49908959970584926, 'k_neighbors': 4, 'nearMissVersion': 3, 'downsampling_strategy': 0.37167016284197074, 'n_neighbors_nm': 6, 'n_neighbors_ver3_nm': 18, 'combo_oversampler': 'SVMSMOTE', 'combo_undersampler': 'AllKNN', 'combo_sequence': 'UpDown'}. Best is trial 0 with value: 0.0.


Shape of y: (303268,)
Shape of all_synthetic_labels: (2000,)
Shape of transformed data due to auto encoder = (305268, 29)
    Single_Label Binary_Classification problem 
    time taken to run entire featurewiz = 13140 second(s)
Recursive XGBoost selected 29 features...
#### Starting featurewiz transform for test data ####
Loaded input data. Shape = (75948, 29)
#### Starting lazytransform for test data ####
    Time taken to transform dataset = 1 second(s)
    Shape of transformed dataset: (75948, 29)
Features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
Erro no fold 2: The specified ratio required to remove samples from the minority class while trying to generate new samples. Please increase the ratio.
Tamanho inicial antes do loop: X_train=(227845, 29), y_train=(227845,)
featurewiz is given 0.6 as correlation limit...
    moving targ

... (44 outputs left out)

[I 2025-08-24 15:52:11,683] Trial 31 finished with value: 0.0 and parameters: {'rm_outliers': False, 'apply_fw': False, 'balance_strategy': 'SMOTE', 'sampling_strategy': 0.861198291560112, 'k_neighbors': 6, 'nearMissVersion': 2, 'downsampling_strategy': 0.5482975006099188, 'n_neighbors_nm': 18, 'n_neighbors_ver3_nm': 4, 'combo_oversampler': 'SMOTE', 'combo_undersampler': 'AllKNN', 'classifier': 'LOF', 'n_neighbors': 29, 'leaf_size': 50, 'p': 5}. Best is trial 23 with value: 0.9043660741509818.


Fold 2: Classe majoritária não prevista


[I 2025-08-24 15:52:12,071] Trial 32 finished with value: 0.0 and parameters: {'rm_outliers': False, 'apply_fw': False, 'balance_strategy': 'Balanced', 'sampling_strategy': 0.7325097174925179, 'k_neighbors': 10, 'nearMissVersion': 3, 'downsampling_strategy': 0.02774055746906351, 'n_neighbors_nm': 20, 'n_neighbors_ver3_nm': 7, 'combo_oversampler': 'KMeansSMOTE', 'combo_undersampler': 'Cluster Centroids', 'k_neighbours': 3, 'n_neighbors': 5, 'num_rows': 583, 'epochs': 297}. Best is trial 23 with value: 0.9043660741509818.


Tamanho inicial antes do loop: X_train=(227845, 29), y_train=(227845,)
Erro no fold 0: CategoricalDistribution does not support dynamic value space.
Erro no fold 1: CategoricalDistribution does not support dynamic value space.
Erro no fold 2: CategoricalDistribution does not support dynamic value space.
Tamanho inicial antes do loop: X_train=(227845, 29), y_train=(227845,)
Distribuição após balanceamento: [151634 146005]
